# Intialize

In [ ]:
# init
from importlib import reload
from pathlib import Path

import toolsGeneral.logger as tgl
import toolsGeneral.main as tgm
import toolsPandas.helpers as tgh
import toolsOSM.overpass as too

def pckgs_reload():
    reload(tgm)
    reload(tgl)
    reload(too)
    reload(tgh)

pckgs_reload()

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data"
SAVE_DIR = DATA_DIR / 'raw/osm countries queries'

In [5]:
# %reload_ext auto_init

# Load data to scrape

In [3]:
# init
raw_scrape_logger = tgl.initiate_logger('raw_scrape_logger', DATA_DIR / '/raw/raw_scrape.log')

In [1]:
# init
osmMetaCountrDict = tgm.load(DATA_DIR / "osmMetaCountrDict.json")
print(len(osmMetaCountrDict))

241
241
241


In [7]:
# init
tuples = sorted(
[(k, v["id"], v["addLvlsNum"]) for (k, v) in osmMetaCountrDict.items()],
    key=lambda arg: arg[0]
)
print(len(tuples))

241


# Collect

### exclude processed countries

In [35]:
processed_countries = tgm.load(SAVE_DIR / "processed_countries.pkl")
print(len(processed_countries))

98


In [3]:
to_scrape = [t for t in tuples if t[0] not in processed_countries]
print(len(to_scrape))
to_scrape = to_scrape[:2]
to_scrape = [('China', '270056', ['4', '6', '8'])]

145


In [4]:
to_scrape

[('China', '270056', ['4', '6', '8'])]

### make call

In [6]:
failed_file = SAVE_DIR / 'failed_countries.pkl'
failed_countries = tgm.load(failed_file) if os.path.exists(failed_file) else set()

processed_file = SAVE_DIR / 'processed_countries.pkl'
processed_countries = tgm.load(processed_file) if os.path.exists(processed_file) else set()

In [7]:
raw_scrape_logger.info(f"* processed_countries: {len(processed_countries)}")
raw_scrape_logger.info(f"* failed_countries: {len(failed_countries)}")

[INFO] : * processed_countries: 96
[INFO] : * failed_countries: 0


In [12]:
query = f"""
        [timeout:60][out:json];

	rel(270056);
     out tags;
     map_to_area;
     rel[boundary=administrative][admin_level=4](area);
     out tags;

out geom;
"""

In [13]:
response = requests.get("http://overpass-api.de/api/interpreter", params={"data": query}, timeout=6)
response.status_code

200

In [10]:
for country, id, lvls in to_scrape:
    raw_scrape_logger.info(f"* processing: {country, id, lvls}")
    
    country_save_file = SAVE_DIR / country / f'rawOSMRes.json'
    response = too.getOSMIDAddsStruct(id, lvls)
    raw_scrape_logger.info(f"  - finished: {response['status']}")

    if response["status"] == "ok":
        tgm.dump(country_save_file, response["data"])
        processed_countries.add(country)
        tgm.dump(processed_file, processed_countries)
    elif '429' in response["status_type"]:
        raw_scrape_logger.info(f"  - Too many requests error, trying chunks")
        too.getOSMIDAddsStruct_chunks((country, id, lvls), SAVE_DIR)
    else:
        raw_scrape_logger.info(f"  - Failed, saving to failed_countries")
        failed_countries.add(country)
        tgm.dump(failed_file, failed_countries)
    
    time.sleep(3)

raw_scrape_logger.info(f"* processed_countries: {len(processed_countries)}")
raw_scrape_logger.info(f"* failed_countries: {len(failed_countries)}")

[INFO] : * processing: ('China', '270056', ['4', '6', '8'])


KeyboardInterrupt: 

In [ ]:
for country, id, lvls in to_scrape:

    too.getOSMIDAddsStruct_chunks((country, id, lvls), SAVE_DIR)
    time.sleep(3)

# Review

In [2]:
country_dirs = [f for f in (DATA_DIR / 'raw/osm countries queries').glob('*') if f.is_dir()]
print(len(country_dirs))

97


In [3]:
raw_by_cntr = {}
# for chunks and non chunk files
for dir in country_dirs:
    files_elements = [tgm.load(str(f))['elements'] for f in dir.glob('*.json')]
    elements = [ele for list in files_elements for ele in list]
    raw_by_cntr[str(dir.name)] = elements

print(f"* Number of countries with raw data: {len(raw_by_cntr)}")

* Number of countries with raw data: 97


In [17]:
resume = {}
for country, raw in raw_by_cntr.items():
    resume[country] = {}
    resume[country]['lvl_1'] = len([ele for ele in raw if ele['tags']['admin_level'] == '4'])
    resume[country]['lvl_2'] = len([ele for ele in raw if ele['tags']['admin_level'] == '6'])
    resume[country]['lvl_3'] = len([ele for ele in raw if ele['tags']['admin_level'] == '8'])
    resume[country]['total'] = len([ele for ele in raw if ele['tags']['admin_level'] != '2'])

resume = pd.DataFrame(resume).T
resume.peek()

,lvl_1,lvl_2,lvl_3,total
Afghanistan,34,0,0,34
AkrotiriAndDhekelia,0,0,0,0
Albania,3,12,374,389
Algeria,58,547,1542,2147
Andorra,0,0,0,0
Angola,19,29,15,63
Anguilla,0,0,0,0
AntiguaAndBarbuda,3,8,0,11
Argentina,25,661,524,1210
Australia,15,600,0,615
